In [17]:
from bs4 import BeautifulSoup, SoupStrainer
import requests
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from urllib.parse import urljoin, urlparse


In [15]:
parse_only = SoupStrainer('div', class_='document')

loader = WebBaseLoader(
    web_paths=("https://docs.manim.community/en/stable/",),
    bs_kwargs=dict(parse_only=parse_only)
)

docs = loader.load()

In [16]:
docs

[Document(page_content='', metadata={'source': 'https://docs.manim.community/en/stable/'})]

In [18]:
def get_all_links(url, base_url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(url, href)
        if full_url.startswith(base_url) and full_url != url:
            yield full_url

def crawl_site(start_url, base_url):
    visited = set()
    to_visit = [start_url]
    
    while to_visit:
        current_url = to_visit.pop(0)
        if current_url not in visited and current_url.startswith(base_url):
            print(f"Crawling: {current_url}")
            visited.add(current_url)
            to_visit.extend(link for link in get_all_links(current_url, base_url) if link not in visited)
    
    return list(visited)


In [28]:
def scrape_manim_docs(start_url):
    # base_url = f"{urlparse(start_url).scheme}://{urlparse(start_url).netloc}"
    all_urls = crawl_site(start_url, start_url)
    
    # Define SoupStrainer to parse only the main content
    parse_only = SoupStrainer('div', class_='document')
    
    loader = WebBaseLoader(
        web_paths=all_urls,
        bs_kwargs={"parse_only": parse_only}
    )
    documents = loader.load()
    
    # Split documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    
    split_docs = text_splitter.split_documents(documents)
    return split_docs

In [30]:
docs = scrape_manim_docs("https://docs.manim.community/en/stable/tutorials/")

Crawling: https://docs.manim.community/en/stable/tutorials/
Crawling: https://docs.manim.community/en/stable/tutorials/#furo-main-content
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html
Crawling: https://docs.manim.community/en/stable/tutorials/output_and_config.html
Crawling: https://docs.manim.community/en/stable/tutorials/building_blocks.html
Crawling: https://docs.manim.community/en/stable/tutorials/#tutorials
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html#overview
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html#starting-a-new-project
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html#animating-a-circle
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html#transforming-a-square-into-a-circle
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html#positioning-mobjects
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.ht